In [ ]:
# Deepfake Detection Training Script
# This notebook handles the training of a hybrid ResNext50 + LSTM model.
# ResNext50 extracts spatial features from frames, while LSTM processes 
# the temporal sequences to detect manipulation over time.

# Requirement: GPU should be enabled for training.
# Load the dataset and save checkpoints.

In [ ]:
# Utilities for video validation, frame extraction, and data augmentation.
import glob # For file pattern matching
import torch # Main PyTorch library
import torchvision # Computer vision tools for PyTorch
from torchvision import transforms # Image transformation utilities
from torch.utils.data import DataLoader # Data batching and shuffling
from torch.utils.data.dataset import Dataset # Base class for custom datasets
import os # Operating system interfaces (paths, directories)
import numpy as np # Numerical operations and array handling
import cv2 # OpenCV for video and image processing
import matplotlib.pyplot as plt # Plotting and visualization
import face_recognition # Facial recognition and landmarking

# Validation function to check if a video file can be properly decoded and transformed.
def validate_video(vid_path, train_transforms):
      transform = train_transforms # Store the transform pipeline
      count = 20 # Number of frames to check for validity
      video_path = vid_path # Path to the video file
      frames = [] # List to store valid frames
      for i, frame in enumerate(frame_extract(video_path)): # Iterate through extracted frames
        frames.append(transform(frame)) # Apply transformation to each frame
        if(len(frames) == count): # Stop after reaching the required count
          break
      frames = torch.stack(frames) # Stack individual frames into a single tensor
      frames = frames[:count] # Ensure we only have the desired number of frames
      return frames # Return the validated frame tensor

# Generator to yield frames from a video file.
def frame_extract(path):
  vidObj = cv2.VideoCapture(path) # Open the video file using OpenCV
  success = 1 # Flag to track successful frame read
  while success: # Loop while frames are still being read successfully
      success, image = vidObj.read() # Read the next frame
      if success: # If a frame was successfully read
          yield image # Yield the frame to the caller

# Normalization parameters (ImageNet defaults)
im_size = 112 # Standard input size for the model
mean = [0.485, 0.456, 0.406] # RGB mean values for normalization
std = [0.229, 0.224, 0.225] # RGB standard deviation for normalization

# Transform pipeline: Convert to PIL -> Resize -> Tensor -> Normalize
train_transforms = transforms.Compose([
                                        transforms.ToPILImage(), # Convert NumPy array to PIL Image
                                        transforms.Resize((im_size, im_size)), # Resize to target dimensions
                                        transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
                                        transforms.Normalize(mean, std)]) # Apply ImageNet normalization

# Load paths for all face-only datasets
video_fil =  glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Celeb_fake_face_only/*.mp4') # Load Celeb-DF Fakes
video_fil += glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Celeb_real_face_only/*.mp4') # Load Celeb-DF Reals
video_fil += glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/DFDC_FAKE_Face_only_data/*.mp4') # Load DFDC Fakes
video_fil += glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/DFDC_REAL_Face_only_data/*.mp4') # Load DFDC Reals
video_fil += glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/FF_Face_only_data/*.mp4') # Load FaceForensics++ data

print("Total no of videos :", len(video_fil)) # Output the total count of video files found

# Run validation on all videos to identify corrupted files
count = 0 # Initialize counter
for i in video_fil: # Iterate through each video path
  try:
    count += 1 # Increment processed count
    validate_video(i, train_transforms) # Attempt to validate the video
  except: # If an error occurs (e.g., file cannot be opened)
    print("Corrupted video is : ", i) # Report the corrupted file
    continue # Move to the next video


In [ ]:
# Dataset Loading and Preparation
import json # For handling JSON metadata if needed
import random # For shuffling data

video_files = video_fil # Assign validated files to working variable
random.shuffle(video_files) # Shuffle to mix real and fake samples for better training

# Custom Dataset class to handle sequence loading
class video_dataset(Dataset):
    def __init__(self, video_names, labels, sequence_length=60, transform=None):
        self.video_names = video_names # List of video paths
        self.labels = labels # Dataframe containing file-to-label mapping
        self.transform = transform # Transformation pipeline
        self.count = sequence_length # Number of frames to extract per video

    def __len__(self):
        return len(self.video_names) # Return the total number of videos in the dataset

    def __getitem__(self, idx):
        video_path = self.video_names[idx] # Get path for the current index
        frames = [] # List to store frames for this video
        temp_video = video_path.split('/')[-1] # Extract just the filename from the path
        
        # Look up label in the metadata CSV
        label = self.labels.iloc[(labels.loc[labels["file"] == temp_video].index.values[0]), 1] # Find label based on filename
        label = 0 if label == 'FAKE' else 1 # Map 'FAKE' to 0 and others (REAL) to 1
        
        # Extract the required number of frames
        for i, frame in enumerate(self.frame_extract(video_path)): # Iterate through frames
          frames.append(self.transform(frame)) # Apply transforms and add to list
          if(len(frames) == self.count): # Stop when we have enough frames for the sequence
            break
        
        frames = torch.stack(frames) # Stack frames into a sequence tensor (seq_len, C, H, W)
        frames = frames[:self.count] # Truncate if there are extra frames
        return frames, label # Return the sequence of frames and its label

    def frame_extract(self, path):
      vidObj = cv2.VideoCapture(path) # Open video file
      success = 1 # Success flag
      while success: # Loop through frames
          success, image = vidObj.read() # Read frame
          if success: # If successful
              yield image # Yield frame


In [ ]:
# Model Definition: ResNext + LSTM
from torch import nn # Neural network modules
from torchvision import models # Pre-trained computer vision models

class Model(nn.Module):
    def __init__(self, num_classes, latent_dim=2048, lstm_layers=1, hidden_dim=2048, bidirectional=False):
        super(Model, self).__init__()
        # Pre-trained ResNext50 for spatial feature extraction
        model = models.resnext50_32x4d(pretrained=True) # Load pre-trained ResNext
        self.model = nn.Sequential(*list(model.children())[:-2]) # Remove fully connected layers
        # LSTM for temporal feature extraction
        self.lstm = nn.LSTM(latent_dim, hidden_dim, lstm_layers, bidirectional) # Initialize LSTM
        self.relu = nn.LeakyReLU() # Non-linear activation
        self.dp = nn.Dropout(0.4) # Dropout for regularization
        self.linear1 = nn.Linear(2048, num_classes) # Final classification layer
        self.avgpool = nn.AdaptiveAvgPool2d(1) # Global average pooling

    def forward(self, x):
        batch_size, seq_length, c, h, w = x.shape # Unpack input dimensions
        # Process every frame in the sequence through the CNN
        x = x.view(batch_size * seq_length, c, h, w) # Reshape for parallel CNN processing
        fmap = self.model(x) # Extract spatial features using ResNext
        x = self.avgpool(fmap) # Apply pooling
        x = x.view(batch_size, seq_length, 2048) # Reshape for LSTM (batch, sequence, features)
        # Process the sequence of features through the LSTM
        x_lstm, _ = self.lstm(x, None) # Temporal processing
        # Average the LSTM outputs over the sequence for final classification
        return fmap, self.dp(self.linear1(torch.mean(x_lstm, dim=1))) # Classification output

In [ ]:
model = Model(2).cuda() # Instantiate model for 2 classes and move to GPU
# Test with dummy input
a,b = model(torch.from_numpy(np.empty((1,20,3,112,112))).type(torch.cuda.FloatTensor)) # Verify forward pass

In [ ]:
import torch # Redundant import for clarity in cell
from torch.autograd import Variable # For automatic differentiation
import time # For tracking time
import os # For path handling
import sys # For standard output manipulation

def train_epoch(epoch, num_epochs, data_loader, model, criterion, optimizer):
    model.train() # Set model to training mode
    losses = AverageMeter() # Track training loss
    accuracies = AverageMeter() # Track training accuracy
    t = [] # Unused list
    for i, (inputs, targets) in enumerate(data_loader): # Iterate over batches
        if torch.cuda.is_available(): # Check for GPU
            targets = targets.type(torch.cuda.LongTensor) # Move labels to GPU
            inputs = inputs.cuda() # Move images to GPU
        _,outputs = model(inputs) # Forward pass
        loss  = criterion(outputs,targets.type(torch.cuda.LongTensor)) # Calculate loss
        acc = calculate_accuracy(outputs, targets.type(torch.cuda.LongTensor)) # Calculate accuracy
        losses.update(loss.item(), inputs.size(0)) # Update loss stats
        accuracies.update(acc, inputs.size(0)) # Update accuracy stats
        optimizer.zero_grad() # Clear previous gradients
        loss.backward() # Backward pass (compute gradients)
        optimizer.step() # Update weights
        sys.stdout.write( # Progress reporting
                "\r[Epoch %d/%d] [Batch %d / %d] [Loss: %f, Acc: %.2f%%]"
                % (
                    epoch,
                    num_epochs,
                    i,
                    len(data_loader),
                    losses.avg,
                    accuracies.avg))
    torch.save(model.state_dict(),'/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Model Creation/checkpoint.pt') # Save weights to local project directory
    return losses.avg,accuracies.avg # Return epoch metrics

def test(epoch,model, data_loader ,criterion):
    print('Testing')
    model.eval() # Set model to evaluation mode
    losses = AverageMeter() # Track validation loss
    accuracies = AverageMeter() # Track validation accuracy
    pred = [] # Store all predictions
    true = [] # Store all true labels
    count = 0 # Initialize counter
    with torch.no_grad(): # Disable gradient computation for speed
        for i, (inputs, targets) in enumerate(data_loader): # Iterate over test batches
            if torch.cuda.is_available(): # GPU check
                targets = targets.cuda().type(torch.cuda.FloatTensor) # Prepare targets
                inputs = inputs.cuda() # Prepare inputs
            _,outputs = model(inputs) # Forward pass
            loss = torch.mean(criterion(outputs, targets.type(torch.cuda.LongTensor))) # Loss calculation
            acc = calculate_accuracy(outputs,targets.type(torch.cuda.LongTensor)) # Accuracy calculation
            _,p = torch.max(outputs,1) # Get predicted classes
            true += (targets.type(torch.cuda.LongTensor)).detach().cpu().numpy().reshape(len(targets)).tolist() # Store ground truth
            pred += p.detach().cpu().numpy().reshape(len(p)).tolist() # Store predictions
            losses.update(loss.item(), inputs.size(0)) # Update stats
            accuracies.update(acc, inputs.size(0)) # Update stats
            sys.stdout.write( # Progress reporting
                    "\r[Batch %d / %d]  [Loss: %f, Acc: %.2f%%]"
                    % (
                        i,
                        len(data_loader),
                        losses.avg,
                        accuracies.avg
                        )
                    )
        print('\nAccuracy {}'.format(accuracies.avg)) # Final validation accuracy
    return true,pred,losses.avg,accuracies.avg # Return evaluation results

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset() # Initialize values
    def reset(self):
        self.val = 0 # Reset current value
        self.avg = 0 # Reset average
        self.sum = 0 # Reset sum
        self.count = 0 # Reset count

    def update(self, val, n=1):
        self.val = val # Set latest value
        self.sum += val * n # Add to total sum
        self.count += n # Increment count
        self.avg = self.sum / self.count # Recalculate average

def calculate_accuracy(outputs, targets):
    batch_size = targets.size(0) # Get batch size

    _, pred = outputs.topk(1, 1, True) # Get index of max probability
    pred = pred.t() # Transpose prediction tensor
    correct = pred.eq(targets.view(1, -1)) # Check for matches with targets
    n_correct_elems = correct.float().sum().item() # Count correct matches
    return 100* n_correct_elems / batch_size # Return percentage accuracy

In [ ]:
import seaborn as sn # For heatmap visualization
#Output confusion matrix
def print_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred) # Generate confusion matrix array
    print('True positive = ', cm[0][0]) # Output counts
    print('False positive = ', cm[0][1])
    print('False negative = ', cm[1][0])
    print('True negative = ', cm[1][1])
    print('\n')
    df_cm = pd.DataFrame(cm, range(2), range(2)) # Create DataFrame for plotting
    sn.set(font_scale=1.4) # Adjust font scale
    sn.heatmap(df_cm, annot=True, annot_kws={"size": 16}) # Draw heatmap
    plt.ylabel('Actual label', size = 20) # Y-axis label
    plt.xlabel('Predicted label', size = 20) # X-axis label
    plt.xticks(np.arange(2), ['Fake', 'Real'], size = 16) # X-axis ticks
    plt.yticks(np.arange(2), ['Fake', 'Real'], size = 16) # Y-axis ticks
    plt.ylim([2, 0]) # Adjust limits
    plt.show() # Display plot
    calculated_acc = (cm[0][0]+cm[1][1])/(cm[0][0]+cm[0][1]+cm[1][0]+ cm[1][1]) # Manual accuracy check
    print("Calculated Accuracy",calculated_acc*100) # Output calculated accuracy

In [ ]:
def plot_loss(train_loss_avg,test_loss_avg,num_epochs):
  loss_train = train_loss_avg # Training loss history
  loss_val = test_loss_avg # Validation loss history
  print(num_epochs) # Output number of epochs
  epochs = range(1,num_epochs+1) # X-axis range
  plt.plot(epochs, loss_train, 'g', label='Training loss') # Plot training loss
  plt.plot(epochs, loss_val, 'b', label='validation loss') # Plot validation loss
  plt.title('Training and Validation loss') # Title
  plt.xlabel('Epochs') # X-axis label
  plt.ylabel('Loss') # Y-axis label
  plt.legend() # Show legend
  plt.show() # Show plot

def plot_accuracy(train_accuracy,test_accuracy,num_epochs):
  loss_train = train_accuracy # Training accuracy history
  loss_val = test_accuracy # Validation accuracy history
  epochs = range(1,num_epochs+1) # X-axis range
  plt.plot(epochs, loss_train, 'g', label='Training accuracy') # Plot training accuracy
  plt.plot(epochs, loss_val, 'b', label='validation accuracy') # Plot validation accuracy
  plt.title('Training and Validation accuracy') # Title
  plt.xlabel('Epochs') # X-axis label
  plt.ylabel('Accuracy') # Y-axis label
  plt.legend() # Show legend
  plt.show() # Show plot

In [ ]:
from sklearn.metrics import confusion_matrix # Scikit-learn CM utility
#learning rate
lr = 1e-5#0.001 # Set learning rate for Adam optimizer
#number of epochs 
num_epochs = 20 # Total training iterations

optimizer = torch.optim.Adam(model.parameters(), lr= lr,weight_decay = 1e-5) # Initialize Adam optimizer

#class_weights = torch.from_numpy(np.asarray([1,15])).type(torch.FloatTensor).cuda() # Optional: address class imbalance
#criterion = nn.CrossEntropyLoss(weight = class_weights).cuda() # Optional: weighted loss
criterion = nn.CrossEntropyLoss().cuda() # Standard Cross Entropy loss function
train_loss_avg =[] # Initialize metrics lists
train_accuracy = []
test_loss_avg = []
test_accuracy = []

for epoch in range(1,num_epochs+1): # Training loop
    l, acc = train_epoch(epoch,num_epochs,train_loader,model,criterion,optimizer) # Execute one training epoch
    train_loss_avg.append(l) # Store metrics
    train_accuracy.append(acc)
    true,pred,tl,t_acc = test(epoch,model,valid_loader,criterion) # Execute validation
    test_loss_avg.append(tl) # Store metrics
    test_accuracy.append(t_acc)

plot_loss(train_loss_avg,test_loss_avg,len(train_loss_avg)) # Visualize loss trends
plot_accuracy(train_accuracy,test_accuracy,len(train_accuracy)) # Visualize accuracy trends
print(confusion_matrix(true,pred)) # Output CM array
print_confusion_matrix(true,pred) # Draw CM heatmap

In [ ]:
def plot_loss(train_loss_avg,test_loss_avg,num_epochs):
  loss_train = train_loss_avg
  loss_val = test_loss_avg
  print(num_epochs)
  epochs = range(1,num_epochs+1)
  plt.plot(epochs, loss_train, 'g', label='Training loss')
  plt.plot(epochs, loss_val, 'b', label='validation loss')
  plt.title('Training and Validation loss')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.legend()
  plt.show()
def plot_accuracy(train_accuracy,test_accuracy,num_epochs):
  loss_train = train_accuracy
  loss_val = test_accuracy
  epochs = range(1,num_epochs+1)
  plt.plot(epochs, loss_train, 'g', label='Training accuracy')
  plt.plot(epochs, loss_val, 'b', label='validation accuracy')
  plt.title('Training and Validation accuracy')
  plt.xlabel('Epochs')
  plt.ylabel('Accuracy')
  plt.legend()
  plt.show()